# Kvasir polyp centroid tracking under controlled motion

Use frozen Farneback optical flow and phase correlation features to fit a regularized model for centroid displacement. Compare against no motion and median foreground flow. The first frame's ground-truth mask supplies foreground pixels. This is pairwise foreground-centroid prediction with a known starting mask. Use real endoscopy images and annotated polyp masks from Kvasir-SEG. Resize to 160 × 96 and apply controlled affine changes, exposure changes and noise. Hold out source images so no transformed copy of a test image appears in training. Motion is simulated. The dataset passed filters for 1260 pairs from 420 groups.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'deformation-motion-tracking-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'opencv-python-headless==5.1.0.95', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import hashlib\nimport os\nimport zipfile\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\nroot = (\n    Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\n    / "kvasir-seg"\n)\nroot.mkdir(parents=True, exist_ok=True)\ntarget = root / "kvasir-seg.zip"\nif not target.exists():\n    temp = target.with_suffix(".part")\n    urlretrieve("https://datasets.simula.no/downloads/kvasir-seg.zip", temp)\n    temp.replace(target)\nassert (\n    hashlib.sha256(target.read_bytes()).hexdigest()\n    == "03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7"\n)\nwith zipfile.ZipFile(target) as archive:\n    archive.extractall(root / "extracted")\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nimport cv2\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import GroupShuffleSplit\n\ncv2.setNumThreads(1)\n\n\ndef center(mask):\n    y, x = np.where(mask)\n    return np.array([x.mean(), y.mean()])\n\n\ndef features(a, b, mask):\n    flow = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 19, 3, 5, 1.2, 0)\n    inside = flow[mask]\n    outside = flow[~mask]\n    local = np.median(inside, axis=0)\n    shift, _ = cv2.phaseCorrelate(a.astype("float32"), b.astype("float32"))\n    return np.r_[\n        local,\n        inside.mean(0),\n        np.quantile(inside, 0.25, axis=0),\n        np.quantile(inside, 0.75, axis=0),\n        np.median(outside, axis=0),\n        shift,\n    ]\n\n\ndef example(rows, a, b, m, n, group):\n    if 30 <= m.sum() < m.size * 0.9 and n.sum() >= 30:\n        rows.append((features(a, b, m), center(n) - center(m), group))\n\n\ndef public():\n    root = DATA / "kvasir-seg" / "extracted" / "Kvasir-SEG"\n    paths = sorted((root / "images").glob("*.jpg"))\n    assert len(paths) == 1000\n    rng = np.random.default_rng(117)\n    selected = np.sort(rng.choice(len(paths), 420, replace=False))\n    rows = []\n    for index in selected:\n        path = paths[index]\n        a = cv2.resize(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), (160, 96))\n        m = (\n            cv2.resize(\n                cv2.imread(str(root / "masks" / path.name), cv2.IMREAD_GRAYSCALE),\n                (160, 96),\n                interpolation=cv2.INTER_NEAREST,\n            )\n            > 127\n        )\n        for _ in range(3):\n            x, y = (float(z) for z in center(m))\n            mat = cv2.getRotationMatrix2D((x, y), rng.normal(0, 8), 1.0)\n            mat[:, 0] *= rng.uniform(0.87, 1.13)\n            mat[:, 2] += rng.normal(0, 5, 2)\n            b = cv2.warpAffine(a, mat, (160, 96), borderMode=cv2.BORDER_REFLECT)\n            b = np.clip(\n                b * rng.uniform(0.68, 1.25) + rng.normal(0, 6, b.shape), 0, 255\n            ).astype("uint8")\n            n = (\n                cv2.warpAffine(\n                    m.astype("uint8"), mat, (160, 96), flags=cv2.INTER_NEAREST\n                )\n                > 0\n            )\n            example(rows, a, b, m, n, path.stem)\n    assert len(rows) >= 800\n    return rows\n\n\ndef run(seed):\n    x, y, g = observations\n    train, test = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.28, random_state=seed).split(x, y, g)\n    )\n    assert not set(g[train]) & set(g[test])\n    model = make_pipeline(StandardScaler(), Ridge(alpha=18)).fit(x[train], y[train])\n    save((model, np.unique(g[test])), f"ridge-{seed}")\n    return {\n        method: {\n            "mean_error_px": float(np.linalg.norm(pred - y[test], axis=1).mean()),\n            "median_error_px": float(np.median(np.linalg.norm(pred - y[test], axis=1))),\n            "test_sequences": len(np.unique(g[test])),\n            "test_pairs": len(test),\n        }\n        for method, pred in [\n            ("stationary", np.zeros_like(y[test])),\n            ("median_flow", x[test, :2]),\n            ("learned_fusion", model.predict(x[test])),\n        ]\n    }\n\n\nif __name__ == "__main__":\n    rows = public()\n    x, y, g = zip(*rows)\n    observations = (np.stack(x), np.stack(y), np.array(g))\n    runs = {seed: run(seed) for seed in SEEDS}\n    means = {\n        method: {\n            key: float(np.mean([run[method][key] for run in runs.values()]))\n            for key in metrics\n        }\n        for method, metrics in runs[SEEDS[0]].items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps(\n            {\n                "seeds": SEEDS,\n                "runs": runs,\n                "means": means,\n                "samples": len(g),\n                "sequences": len(set(g)),\n            },\n            indent=2,\n        )\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())